# 08 Judge calibration
Check an LLM judge against 40 human labels before you trust its scores.

Step 1: set up the kit

In [1]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork (and BRANCH).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
BRANCH = "main"  # change to your branch name if you are working on a branch
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-08"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Kit folder found: atp-builder-kit


Step 2: load the human labels

In [2]:
import re
import pandas as pd
from kit.llm import call_llm
from kit.prices import cost

labels = pd.read_csv(ROOT / "evals" / "labels.csv")
print(labels.groupby(["split", "human_label"]).size().unstack())
labels.head(3)

human_label  fail  pass
split                  
dev             8    12
held-out        9    11


,id,split,question,reference,answer,human_label,note
0,l01,dev,How do I export a note as PDF?,"Open the note, select More or the three-dot me...","Open the note, choose More, then Export, pick ...",pass,correct
1,l02,dev,How do I export a note as PDF?,"Open the note, select More or the three-dot me...",Go to Settings and turn on PDF sync. Your note...,fail,invented feature
2,l03,dev,Can I get a deleted note back?,"Open Trash in the sidebar, find the note, open...","Yes. Open Trash in the sidebar, find the note ...",pass,correct


Step 3: the judge prompt

In [3]:
JUDGE_SYSTEM = ("You grade answers from a help assistant. PASS only if the answer is correct and fully "
                "supported by the reference. Any wrong detail, invented feature or refusal is a FAIL. "
                "Reply with PASS or FAIL, a colon, then one short reason.")

def judge(row):
    r = call_llm(f"Question: {row.question}\nReference: {row.reference}\nAnswer: {row.answer}\n"
                 "Reply with PASS or FAIL.", system=JUDGE_SYSTEM, max_output_tokens=800)
    verdict = re.match(r"\s*(PASS|FAIL)", r["text"].upper())
    return (verdict.group(1).lower() if verdict else "unparsed"), r["text"], cost(r["input_tokens"], r["output_tokens"], r["model"])

sys.path.insert(0, str(ROOT / "evals"))
from gates import JUDGE_SYSTEM as SUITE_JUDGE  # the judge the eval suite runs (evals/gates.py)
print("Same judge prompt as the eval suite:", JUDGE_SYSTEM == SUITE_JUDGE)
print(judge(labels.iloc[0]))

Same judge prompt as the eval suite: True
('pass', 'PASS: matches the reference.', 0.000165)


Step 4: run the judge on every label

In [4]:
out = [judge(row) for row in labels.itertuples()]
labels["judge_label"] = [o[0] for o in out]
labels["judge_reason"] = [o[1] for o in out]
print("Judge cost for 40 labels: $", round(sum(o[2] for o in out), 5))
print("Unparsed verdicts:", int((labels.judge_label == "unparsed").sum()))

Judge cost for 40 labels: $ 0.00648
Unparsed verdicts: 0


Step 5: confusion matrix and rates, dev and held-out

In [5]:
def rates(df):
    tp = ((df.human_label == "pass") & (df.judge_label == "pass")).sum()
    tn = ((df.human_label == "fail") & (df.judge_label == "fail")).sum()
    return {"true_positive_rate": tp / (df.human_label == "pass").sum(),
            "true_negative_rate": tn / (df.human_label == "fail").sum(),
            "agreement": (df.human_label == df.judge_label).mean()}

calibration = {}
for split in ["dev", "held-out"]:
    part = labels[labels.split == split]
    print(f"\n{split}:")
    print(pd.crosstab(part.human_label, part.judge_label, rownames=["human"], colnames=["judge"]))
    calibration[split] = {k: round(float(v), 2) for k, v in rates(part).items()} | {"labels": len(part)}
    print(calibration[split])
# Report these two numbers (held-out) in stage-08/README.md. Turn on JUDGE_HARD=1 only if both are high enough.
import json
_ = (OUTPUT / "judge_calibration.json").write_text(json.dumps(calibration, indent=1))
labels.to_csv(OUTPUT / "judge_labels_scored.csv", index=False)
print("\nSaved judge_calibration.json and judge_labels_scored.csv")


dev:
judge  fail  pass
human            
fail      4     4
pass      0    12
{'true_positive_rate': 1.0, 'true_negative_rate': 0.5, 'agreement': 0.8, 'labels': 20}

held-out:
judge  fail  pass
human            
fail      4     5
pass      0    11
{'true_positive_rate': 1.0, 'true_negative_rate': 0.44, 'agreement': 0.75, 'labels': 20}

Saved judge_calibration.json and judge_labels_scored.csv


Step 6: read the held-out disagreements

In [6]:
held = labels[labels.split == "held-out"]
held[held.human_label != held.judge_label][["id", "answer", "human_label", "judge_label", "note"]]

,id,answer,human_label,judge_label,note
21,l22,"Open the note, choose More, then Print, and pi...",fail,pass,wrong format
25,l26,"Check your connection, tap Sync, and confirm y...",fail,pass,"wrong menu, right words"
29,l30,"Plus adds shared notebooks, version history an...",fail,pass,"wrong plan, right words"
33,l34,"Owner, Admin, Editor and Guest. Guests can edi...",fail,pass,invented role
37,l38,"Open Admin Console > Users, select Delete user...",fail,pass,deletion is permanent
